# DataScout LLM server (Kaggle / Colab)

Runs an Ollama model on the notebook GPU and exposes it through a temporary public URL, so your local
DataScout backend can use it (`OLLAMA_BASE_URL=<url>` in `.env`).

**Before running**
- Kaggle: Settings -> Accelerator = GPU (T4/P100), Internet = ON (needs a verified phone number). Colab: Runtime -> GPU.
- The URL has **no authentication**: anyone who has it can use your GPU. Do not share or commit it.
- The URL changes every session. Session length and GPU quota are limited, so check the current limits.
- Only column summaries are sent by DataScout (no raw rows), but do not use confidential data with a public tunnel anyway.

Run the cells top to bottom and leave the last cell running while you use the app.

In [ ]:
MODEL = "qwen2.5:7b"     # fits a 16 GB GPU

## 1. Install Ollama

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

## 2. Start the server and pull the model

In [ ]:
import os, subprocess, time, urllib.request

server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama did not start")
print("ollama is up")
!ollama pull {MODEL}
!ollama list

## 3. Open the public tunnel (cloudflared, no account needed)

In [ ]:
import re
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

# log to a file (an undrained pipe would eventually block the tunnel)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--no-autoupdate"],
                          stdout=open("cloudflared.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if m:
        url = m.group(0); break
    time.sleep(1)
if not url:
    raise RuntimeError("no tunnel URL; see cloudflared.log")
print("\nPut this in your local .env:\n")
print("DATASCOUT_LLM_PROVIDER=ollama")
print(f"DATASCOUT_LLM_MODEL={MODEL}")
print(f"OLLAMA_BASE_URL={url}")


## 4. Check the tunnel from inside the notebook

In [ ]:
import json
print(json.load(urllib.request.urlopen(f"{url}/api/tags", timeout=30)))

## 5. Keep alive

Leave this cell running. Stop it (or the session) to shut the server down.
On your computer run `python scripts/check_llm.py` to confirm the connection.

In [ ]:
while True:
    time.sleep(60)
    if server.poll() is not None or tunnel.poll() is not None:
        print("server or tunnel stopped, re-run the cells above"); break